# Phase 2: Training LightGBM on the Imbalanced Dataset

This notebook trains a `lightgbm.LGBMClassifier` on our imbalanced heading dataset. Heading classes such as `TITLE`, `H3` and `H4` are rare compared to `BODY_TEXT`, so the training data has to be balanced one way or another. The strategy is chosen with the `BALANCING` setting:

- **`class_weight`**: LightGBM's built-in balancer (`is_unbalance=True`), which reweights the classes during training.
- **`smote`**: the **Synthetic Minority Over-sampling Technique (SMOTE)** creates synthetic minority samples in the training split.
- **`adasyn`**: **Adaptive Synthetic Sampling (ADASYN)** works like SMOTE but generates more synthetic data for the samples that are harder to learn.

**Goal:** Compare how well each strategy recovers the minority heading classes.

**Steps:**
1. Load the pre-processed dataset.
2. Split the data into training and testing sets.
3. Balance the training data (`smote`/`adasyn` resample the **training data only**; `class_weight` leaves it as is).
4. Train the `LGBMClassifier`.
5. Evaluate the model on the original, untouched test set.

### 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer

# Oversamplers for the smote and adasyn strategies
from imblearn.over_sampling import SMOTE, ADASYN

print("Libraries imported successfully.")

### 2. Choose the Balancing Strategy

Set `BALANCING` here or through the `BALANCING` environment variable.

In [ ]:
# class_weight -> LightGBM's built-in balancer (is_unbalance=True)
# smote        -> SMOTE oversampling of the training split
# adasyn       -> ADASYN oversampling of the training split
BALANCING = os.environ.get('BALANCING', 'smote')

BALANCING_OPTIONS = ('class_weight', 'smote', 'adasyn')
if BALANCING not in BALANCING_OPTIONS:
    raise ValueError(f"BALANCING must be one of {BALANCING_OPTIONS}, got {BALANCING!r}")
print(f"Balancing strategy: {BALANCING}")

### 3. Load and Prepare the Dataset

We load the `final_training_dataset.csv` file created in Phase 1. It contains our 12 engineered features and the corresponding target labels.

In [ ]:
# CSV written by AIH_preprocessor/stage3.ipynb
DATASET_PATH = os.environ.get('DATASET_PATH', '../AIH_preprocessor/final_training_dataset.csv')
df = pd.read_csv(DATASET_PATH)

X = df.drop('label', axis=1)
y = df['label']

# Define the label mapping for later visualization
label_map = {
    0: "BODY_TEXT", 1: "TITLE", 2: "H1",
    3: "H2", 4: "H3", 5: "H4"
}
label_names = [label_map[i] for i in sorted(y.unique())]

print("✅ Dataset loaded and prepared.")
print(f"X shape: {X.shape}, y shape: {y.shape}")

### 4. Split Data into Training and Testing Sets

We use a standard 80/20 split. `stratify=y` keeps the proportion of each class the same in the training and testing sets, which is very important for imbalanced datasets. The split happens *before* any resampling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data split into training and testing sets.")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

### 5. Balance the Training Data

- `class_weight`: nothing to do here, LightGBM calculates and applies class weights during training.
- `smote` / `adasyn`: missing values are imputed first (median, fitted on the training data only), then the minority classes of the **training set only** are oversampled. This prevents data leakage, so the model is still evaluated on real, unseen data. The neighbour count must be smaller than the number of samples in the smallest class, otherwise the oversampler crashes.

In [ ]:
if BALANCING == 'class_weight':
    # LightGBM reweights the classes itself, so the model trains on the split as is
    X_train_resampled, y_train_resampled = X_train, y_train
else:
    print(f"Class distribution before {BALANCING.upper()}:")
    print(y_train.value_counts().sort_index())
    print(f"\nChecking for NaN values in X_train...")
    nan_count = X_train.isnull().sum().sum()
    print(f"Found {nan_count} NaN values in the training data.")

    if nan_count > 0:
        print("Applying imputation to fill missing values...")
        # Initialize the imputer to replace NaNs with the median of each column
        imputer = SimpleImputer(strategy='median')

        # Fit the imputer on the training data and transform it
        X_train_imputed = imputer.fit_transform(X_train)
        # IMPORTANT: We only use .transform on the test data to avoid data leakage
        X_test_imputed = imputer.transform(X_test)

        # Convert the imputed numpy arrays back to pandas DataFrames
        X_train = pd.DataFrame(X_train_imputed, columns=X.columns)
        X_test = pd.DataFrame(X_test_imputed, columns=X.columns)

        print("✅ Missing values handled.")
    else:
        print("✅ No missing values found.")

    # Find the number of samples in the smallest class; the neighbour count must be smaller
    min_samples = y_train.value_counts().min()
    k = min_samples - 1 if min_samples > 1 else 1

    if BALANCING == 'smote':
        print(f"\nSmallest class has {min_samples} samples. Setting k_neighbors for SMOTE to {k}.")
        sampler = SMOTE(random_state=42, k_neighbors=k)
    else:
        print(f"\nSmallest class has {min_samples} samples. Setting n_neighbors for ADASYN to {k}.")
        sampler = ADASYN(random_state=42, n_neighbors=k)

    # Fit and apply the oversampler
    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train, y_train)

    print(f"\nClass distribution after {BALANCING.upper()}:")
    print(y_train_resampled.value_counts().sort_index())

    print(f"\nOriginal training set size: {len(X_train)}")
    print(f"Resampled training set size: {len(X_train_resampled)}")

### 6. Train the LightGBM Model

With `class_weight` the model is initialised with `is_unbalance=True`. After SMOTE or ADASYN the training data is already balanced, so the parameter is **not** needed and the model is trained on the larger, resampled training set.

In [ ]:
if BALANCING == 'class_weight':
    lgbm = lgb.LGBMClassifier(is_unbalance=True, random_state=42)
else:
    lgbm = lgb.LGBMClassifier(random_state=42)

print(f"Training the LightGBM model ({BALANCING})...")
lgbm.fit(X_train_resampled, y_train_resampled)
print("✅ Model training complete.")

### 7. Evaluate Model Performance on the Original Test Set

We evaluate the model on the original, untouched `X_test` and `y_test`, which gives an honest assessment of how well it generalises to real, imbalanced data. We look at:
- **Accuracy**: Overall correctness.
- **Classification Report**: A detailed breakdown of precision, recall, and F1-score for each class.
- **Confusion Matrix**: A visual grid showing what the model predicted versus the actual labels.

In [ ]:
STRATEGY_NAMES = {'class_weight': 'is_unbalance=True', 'smote': 'SMOTE', 'adasyn': 'ADASYN'}

# Make predictions on the original test set
y_pred = lgbm.predict(X_test)

# --- Accuracy Score ---
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.4f}\n")

# --- Classification Report ---
print(f"Classification Report (with {STRATEGY_NAMES[BALANCING]}):")
print(classification_report(y_test, y_pred, target_names=label_names))

# --- Confusion Matrix ---
print(f"Confusion Matrix (with {STRATEGY_NAMES[BALANCING]}):")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Confusion Matrix for LightGBM with {STRATEGY_NAMES[BALANCING]}')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

### 8. Analysis and Next Steps

Run the notebook once per strategy and compare the classification reports. Pay close attention to the **recall** and **F1-score** of the minority classes (`TITLE`, `H1`, `H2`, `H3`, `H4`):

- **Recall** tells you: "Of all the actual `TITLE` spans, what percentage did the model correctly identify?" This is crucial because we want to avoid missing these important headings.
- The **F1-score** provides a balanced measure between precision and recall.
- The confusion matrix shows common misclassifications, e.g. headings predicted as `BODY_TEXT` or `H2` confused with `H3`.

If `recall` for `H3` and `H4` is greater than 0 with one of the oversamplers, continuing with algorithmic tuning is a worthwhile path: fine-tune the best model's hyperparameters and build the final prediction pipeline. If it stays at 0 for every strategy, the feature patterns for these classes are too subtle in the current data; the time is better spent labelling more documents (e.g. from DocBank or DocLayNet) to give the model more examples of the rare classes.